# LUMIN Agents (OpenAI + OLMo)

This notebook wires the LUMIN query compiler to two LLM backends:

- OpenAI (via `OPENAI_API_KEY`)
- OLMo 7B Instruct (OpenAI-compatible endpoint via `OLMO_BASE_URL` and `OLMO_API_KEY`)

In [22]:
# Clone the repo into Colab
!git clone https://github.com/anhkos/LUMIN-v2-Search-Agent.git

fatal: destination path 'LUMIN-v2-Search-Agent' already exists and is not an empty directory.


In [23]:
%pip install -r "/content/LUMIN-v2-Search-Agent/requirements.txt"

In [24]:
import os
import sys

repo_root = "/content/LUMIN-v2-Search-Agent"
sys.path.insert(0, os.path.join(repo_root, "src"))
print("repo_root:", repo_root)

repo_root: /content/LUMIN-v2-Search-Agent


In [25]:
import json
import os
import sys
import ast
from dotenv import load_dotenv  # type: ignore
from openai import OpenAI  # type: ignore

# Auto-detect environment: Colab vs local
if os.path.exists("/content/LUMIN-v2-Search-Agent"):
    repo_root = "/content/LUMIN-v2-Search-Agent"
else:
    repo_root = os.path.abspath(os.path.join(os.path.dirname(__file__) if '__file__' in dir() else os.getcwd(), ".."))
    # Fallback for notebook environment
    if not os.path.exists(os.path.join(repo_root, "data", "ontology.json")):
        repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from logic_engine import NeuroSymbolicSolver

load_dotenv()
ontology_path = os.path.join(repo_root, "data", "ontology.json")
print(f"Using ontology: {ontology_path}")

def _is_concept(obj):
    return isinstance(obj, dict) and "type" in obj and "field" in obj

def _flatten_ontology(raw):
    """Extract concept names and their payloads from grouped ontology."""
    flat = {}
    groups = {}
    for key, val in raw.items():
        if key.startswith("__") and key.endswith("__"):
            if key == "__METADATA__":
                continue
            if isinstance(val, dict):
                group_name = key.strip("_").replace("_", " ").title()
                for concept, payload in val.items():
                    if _is_concept(payload):
                        flat[concept] = payload
                        groups.setdefault(group_name, []).append(concept)
            continue
        if _is_concept(val):
            flat[key] = val
            groups.setdefault("Uncategorized", []).append(key)
    return flat, groups

def _format_ontology_for_prompt(groups, flat):
    """Format ontology for LLM with concept names and descriptions."""
    lines = []
    for group in sorted(groups.keys()):
        lines.append(f"\n## {group}")
        for concept in sorted(groups[group]):
            payload = flat.get(concept, {})
            desc = payload.get("description", "")
            values = payload.get("values", [])
            values_str = ", ".join(values[:5]) + ("..." if len(values) > 5 else "")
            lines.append(f"  - **{concept}**: {desc}")
            if values_str:
                lines.append(f"    Values: {values_str}")
    return "\n".join(lines)

with open(ontology_path, "r") as f:
    raw_ontology = json.load(f)

flat_ontology, ontology_groups = _flatten_ontology(raw_ontology)
ONTOLOGY_TERMS = sorted(flat_ontology.keys())
ONTOLOGY_PROMPT = _format_ontology_for_prompt(ontology_groups, flat_ontology)

print(f"Loaded {len(ONTOLOGY_TERMS)} concepts: {ONTOLOGY_TERMS}")

solver = NeuroSymbolicSolver(ontology_path=flat_ontology)

Using ontology: /content/LUMIN-v2-Search-Agent/data/ontology.json
Loaded 7 concepts: ['Analysis Ready', 'Dust Storm Season', 'Midnight', 'Noon', 'Northern Summer', 'Raw Telemetry', 'Southern Summer']


In [26]:
import math

EMBED_MODEL = "text-embedding-3-small"
EMBED_CACHE_PATH = os.path.join(repo_root, "data", "ontology_embeddings.json")

# Optional manual synonyms for deterministic mapping
SYNONYMS = {
    "hirise": "High Resolution Imaging",
    "hirise images": "High Resolution Imaging",
    "hazcam": "Hazard & Navigation Cameras",
    "navcam": "Hazard & Navigation Cameras",
    "spectrometer": "Spectroscopy (Composition)",
    "spectra": "Spectroscopy (Composition)",
    "radar": "Radar Systems",
    "voyager": "Outer Planets Missions",
    "cassini": "Outer Planets Missions",
    "galileo": "Outer Planets Missions",
    "juno": "Outer Planets Missions",
    "curiosity": "Mars Rovers",
    "perseverance": "Mars Rovers",
    "opportunity": "Mars Rovers",
    "spirit": "Mars Rovers",
    "mars 2020": "Mars Rovers",
    "msl": "Mars Rovers",
    "lunar orbiter": "Lunar Exploration",
    "lcross": "Lunar Exploration",
    "mars": "Mars System",
    "phobos": "Mars System",
    "deimos": "Mars System",
    "saturn": "Saturn System",
    "titan": "Saturn System",
    "jupiter": "Jupiter System",
    "europa": "Jupiter System",
    "calibration": "Calibration Targets",
    "raw images": "Raw Data",
    "calibrated": "Calibrated Data",
}

def _normalize_text(text):
    return " ".join(text.lower().strip().split())

def _cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return dot / (norm_a * norm_b)

def _load_embeddings_cache():
    if os.path.isfile(EMBED_CACHE_PATH):
        with open(EMBED_CACHE_PATH, "r") as f:
            data = json.load(f)
        # Verify cache matches current model AND ontology terms
        if data.get("model") == EMBED_MODEL and set(data.get("terms", [])) == set(ONTOLOGY_TERMS):
            return data
        print("Cache invalidated: ontology terms changed. Will rebuild.")
    return None

def _save_embeddings_cache(terms, vectors):
    data = {"model": EMBED_MODEL, "terms": terms, "vectors": vectors}
    os.makedirs(os.path.dirname(EMBED_CACHE_PATH), exist_ok=True)
    with open(EMBED_CACHE_PATH, "w") as f:
        json.dump(data, f)
    print(f"Saved embeddings cache with {len(terms)} terms.")
    return data

def build_embeddings_cache(force=False):
    if not force:
        cached = _load_embeddings_cache()
        if cached:
            print(f"Using cached embeddings ({len(cached['terms'])} terms)")
            return cached

    print(f"Building embeddings for {len(ONTOLOGY_TERMS)} terms...")
    client = OpenAI()
    terms = list(ONTOLOGY_TERMS)
    resp = client.embeddings.create(model=EMBED_MODEL, input=terms)
    vectors = [item.embedding for item in resp.data]
    return _save_embeddings_cache(terms, vectors)

def select_ontology(term, top_k=5, threshold=0.70):
    """Map a term to the closest ontology concept."""
    term_norm = _normalize_text(term)
    
    # Exact match (case-insensitive)
    for ont_term in solver.ontology:
        if _normalize_text(ont_term) == term_norm:
            return {"ok": True, "term": ont_term, "score": 1.0, "candidates": []}

    # Synonym match
    synonym = SYNONYMS.get(term_norm)
    if synonym and synonym in solver.ontology:
        return {"ok": True, "term": synonym, "score": 1.0, "candidates": []}

    # Embedding match with lowered threshold
    cache = build_embeddings_cache()
    client = OpenAI()
    query_vec = client.embeddings.create(model=EMBED_MODEL, input=[term]).data[0].embedding
    scored = []
    for t, v in zip(cache["terms"], cache["vectors"]):
        scored.append((t, _cosine_similarity(query_vec, v)))
    scored.sort(key=lambda x: x[1], reverse=True)
    candidates = [{"term": t, "score": round(s, 4)} for t, s in scored[:top_k]]
    best_term, best_score = scored[0]
    if best_score >= threshold:
        return {"ok": True, "term": best_term, "score": round(best_score, 4), "candidates": candidates}
    return {"ok": False, "term": None, "score": round(best_score, 4), "candidates": candidates}

In [27]:
SYSTEM_PROMPT = """
You are the LUMIN Query Compiler for NASA PDS.
Your goal: translate a natural-language request into a Logic S-Expression using ONLY the concepts below.

### AVAILABLE CONCEPTS (You MUST select from these):
{ontology_text}

### OPERATIONS:
- INTERSECT(A, B) -> Products matching BOTH concepts
- UNION(A, B) -> Products matching EITHER concept
- DIFFERENCE(Base, Remove) -> Products matching Base but NOT Remove

### CRITICAL RULES:
1. Output ONLY the S-expression or concept name. NO explanations, NO markdown, NO code.
2. You MUST use concept names EXACTLY as written above (e.g., "Mars Rovers" not "Mars rover data")
3. If the query maps to ONE concept, output just that concept name in quotes: "Mars Rovers"
4. If the query needs MULTIPLE concepts, output a tuple: ('INTERSECT', 'High Resolution Imaging', 'Mars Rovers')

### EXAMPLES (follow this exact format):
User: "Images from Curiosity or Perseverance"
Output: "Mars Rovers"

User: "High resolution images from Mars rovers"
Output: ('INTERSECT', 'High Resolution Imaging', 'Mars Rovers')

User: "Saturn moon data but not Titan"
Output: ('DIFFERENCE', 'Saturn System', 'Calibration Targets')

User: "Raw engineering camera data from Mars 2020"
Output: ('INTERSECT', 'Hazard & Navigation Cameras', 'Mars 2020 Raw Images')

User: "Spectroscopy from outer planets"
Output: ('INTERSECT', 'Spectroscopy (Composition)', 'Outer Planets Missions')

User: "Radar data from Mars"
Output: ('INTERSECT', 'Radar Systems', 'Mars System')

### NOW OUTPUT ONLY THE PLAN (no other text):
"""

In [28]:
import re

def _build_prompt():
    return SYSTEM_PROMPT.format(ontology_text=ONTOLOGY_PROMPT)

def _strip_fences(text):
    text = text.strip()
    if text.startswith("```") and text.endswith("```"):
        lines = text.splitlines()
        if len(lines) >= 2:
            return "\n".join(lines[1:-1]).strip()
    return text

def _function_to_tuple(text):
    pattern = r"\b(INTERSECT|UNION|DIFFERENCE)\s*\("
    return re.sub(pattern, r"('\1', ", text)

def _extract_plan_from_verbose(text):
    """Extract S-expression or quoted concept from verbose LLM output."""
    # Look for tuple patterns like ('INTERSECT', 'A', 'B')
    tuple_match = re.search(r"\(\s*['\"]?(INTERSECT|UNION|DIFFERENCE)['\"]?\s*,\s*['\"][^'\"]+['\"]", text)
    if tuple_match:
        # Find the full tuple starting from this match
        start = tuple_match.start()
        depth = 0
        for i, c in enumerate(text[start:]):
            if c == '(':
                depth += 1
            elif c == ')':
                depth -= 1
                if depth == 0:
                    return text[start:start + i + 1]
    
    # Look for function-style like INTERSECT('A', 'B')
    func_match = re.search(r"(INTERSECT|UNION|DIFFERENCE)\s*\([^)]+\)", text)
    if func_match:
        return func_match.group(0)
    
    # Look for a quoted ontology term
    for term in ONTOLOGY_TERMS:
        if f'"{term}"' in text or f"'{term}'" in text:
            return term
    
    # Look for any quoted string that might be a concept
    quoted = re.findall(r'"([^"]+)"|\'([^\']+)\'', text)
    for match in quoted:
        found = match[0] or match[1]
        # Check if it's close to an ontology term
        if found and len(found) > 3:
            return found
    
    return None

def _parse_simple_op(text):
    cleaned = text.strip()
    if cleaned.startswith("(") and cleaned.endswith(")"):
        cleaned = cleaned[1:-1].strip()
    m = re.match(r"^(AND|INTERSECT|UNION|DIFFERENCE)\s*\((.*)\)\s*$", cleaned, re.IGNORECASE)
    if not m:
        return None
    op = m.group(1).upper()
    if op == "AND":
        op = "INTERSECT"
    args = [a.strip().strip("'\"") for a in m.group(2).split(",")]
    if len(args) == 2:
        return (op, args[0], args[1])
    if len(args) > 2:
        # Nested: fold into left-associative expression
        result = (op, args[0], args[1])
        for extra in args[2:]:
            result = (op, result, extra)
        return result
    return None

def _parse_plan(raw_plan):
    cleaned = _strip_fences(raw_plan).strip()
    
    # Direct ontology match
    if cleaned in solver.ontology:
        return cleaned
    
    # Check if it's wrapped in quotes
    if (cleaned.startswith('"') and cleaned.endswith('"')) or (cleaned.startswith("'") and cleaned.endswith("'")):
        inner = cleaned[1:-1]
        if inner in solver.ontology:
            return inner
        return inner  # Return anyway, will be resolved later
    
    # If the response is verbose/explanatory, try to extract the plan
    if len(cleaned) > 100 or '\n' in cleaned:
        extracted = _extract_plan_from_verbose(cleaned)
        if extracted:
            cleaned = extracted
    
    # Simple AND/INTERSECT without quotes
    simple = _parse_simple_op(cleaned)
    if simple is not None:
        return simple
    
    # Plain phrase (word-based, possibly with special chars in ontology terms)
    if re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9\s&()_-]*", cleaned):
        return cleaned
    
    try:
        return ast.literal_eval(cleaned)
    except Exception:
        converted = _function_to_tuple(cleaned)
        try:
            return ast.literal_eval(converted)
        except Exception:
            # Last resort: return as-is for resolution
            return cleaned

def _field_for_term(term):
    if isinstance(term, str):
        return solver.ontology.get(term, {}).get('field')
    return None

def _resolve_terms(plan, unresolved):
    if isinstance(plan, str):
        if plan in solver.ontology:
            return plan
        match = select_ontology(plan)
        if match.get("ok") and match.get("term"):
            return match["term"]
        unresolved.append({"term": plan, "candidates": match.get("candidates", [])})
        return plan
    if isinstance(plan, tuple):
        return tuple(_resolve_terms(item, unresolved) for item in plan)
    return plan

def _normalize_plan(plan):
    if isinstance(plan, str):
        return plan
    if isinstance(plan, tuple):
        if len(plan) == 2:
            return ('UNION', _normalize_plan(plan[0]), _normalize_plan(plan[1]))
        if len(plan) != 3:
            return plan
        op, arg1, arg2 = plan
        arg1 = _normalize_plan(arg1)
        arg2 = _normalize_plan(arg2)
        if op == 'INTERSECT':
            field1 = _field_for_term(arg1)
            field2 = _field_for_term(arg2)
            if field1 and field2 and field1 != field2:
                return ('UNION', arg1, arg2)
        return (op, arg1, arg2)
    return plan

def _exec_plan(plan):
    try:
        result = solver.execute_plan(plan)
        return True, result, None
    except Exception as exc:
        return False, None, str(exc)

def run_agent(user_query, client, model):
    prompt = _build_prompt()
    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': prompt},
            {'role': 'user', 'content': user_query}
        ],
        temperature=0.0,
    )
    raw_plan = response.choices[0].message.content.strip()
    try:
        parsed_plan = _parse_plan(raw_plan)
        unresolved = []
        resolved_plan = _resolve_terms(parsed_plan, unresolved)
        normalized_plan = _normalize_plan(resolved_plan)
    except Exception as exc:
        diagnostics = {
            "error": f"parse_error: {exc}",
            "raw_plan": raw_plan
        }
        return raw_plan, diagnostics

    ok, result, error = _exec_plan(normalized_plan)
    if ok:
        if unresolved:
            return raw_plan, {"result": result, "unresolved": unresolved}
        return raw_plan, result

    diagnostics = {
        "error": error,
        "raw_plan": raw_plan,
        "parsed_plan": str(parsed_plan),
        "resolved_plan": str(resolved_plan),
        "normalized_plan": str(normalized_plan),
        "unresolved": unresolved
    }
    return raw_plan, diagnostics

In [29]:
def openai_agent(user_query, model='gpt-4o'):
    client = OpenAI()
    return run_agent(user_query, client, model)

def olmo_agent(user_query, model='allenai/olmo-3-7b-instruct'):
    api_key = os.getenv('OPENROUTER_API_KEY')
    if not api_key:
        raise ValueError('Missing OPENROUTER_API_KEY env var')
    
    client = OpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=api_key,
        default_headers={
            'HTTP-Referer': os.getenv('OPENROUTER_SITE_URL', ''),
            'X-Title': os.getenv('OPENROUTER_SITE_NAME', '')
        }
    )
    
    return run_agent(user_query, client, model)

## Example Usage

In [30]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter OPENROUTER_API_KEY: ")

In [31]:
# Force rebuild embeddings cache for the new ontology
build_embeddings_cache(force=True)

queries = [
    "Show me high resolution images from the Mars Rovers",
    "Find spectroscopy data for the Saturn System",
    "Retrieve raw data from Perseverance",
    "Search for radar data from Mars",
    "Show me hazard camera images",
    "Find data from outer planets missions",
    "Retrieve calibrated data",
    "Show me lunar exploration data",
    "Find calibration targets",
    "Raw images from Mars 2020"
]

print("="*60)
print("TESTING WITH UPDATED ONTOLOGY")
print(f"Available concepts: {ONTOLOGY_TERMS}")
print("="*60)

for query in queries:
    print(f"\n--- Query: {query} ---")
    
    # OpenAI
    openai_plan, openai_result = openai_agent(query)
    print(f"OpenAI Plan: {openai_plan[:100]}...")
    if isinstance(openai_result, dict) and "error" in openai_result:
        print(f"OpenAI ERROR: {openai_result.get('error')}")
        if openai_result.get('unresolved'):
            print(f"  Unresolved: {openai_result['unresolved']}")
    else:
        print(f"OpenAI Result: {json.dumps(openai_result, indent=2)[:200]}...")
    
    # OLMo
    olmo_plan, olmo_result = olmo_agent(query)
    print(f"OLMo Plan: {olmo_plan[:100]}...")
    if isinstance(olmo_result, dict) and "error" in olmo_result:
        print(f"OLMo ERROR: {olmo_result.get('error')}")
    else:
        print(f"OLMo Result: {json.dumps(olmo_result, indent=2)[:200]}...")

Building embeddings for 7 terms...
Saved embeddings cache with 7 terms.
TESTING WITH UPDATED ONTOLOGY
Available concepts: ['Analysis Ready', 'Dust Storm Season', 'Midnight', 'Noon', 'Northern Summer', 'Raw Telemetry', 'Southern Summer']

--- Query: Show me high resolution images from the Mars Rovers ---
Using cached embeddings (7 terms)
Using cached embeddings (7 terms)
Using cached embeddings (7 terms)
OpenAI Plan: ('INTERSECT', 'High Resolution Imaging', 'Mars Rovers')...
OpenAI ERROR: Concept 'High Resolution Imaging' not found in Ontology.
  Unresolved: [{'term': 'INTERSECT', 'candidates': [{'term': 'Analysis Ready', 'score': 0.2602}, {'term': 'Midnight', 'score': 0.2416}, {'term': 'Northern Summer', 'score': 0.2324}, {'term': 'Raw Telemetry', 'score': 0.207}, {'term': 'Noon', 'score': 0.2001}]}, {'term': 'High Resolution Imaging', 'candidates': [{'term': 'Raw Telemetry', 'score': 0.3294}, {'term': 'Analysis Ready', 'score': 0.2529}, {'term': 'Midnight', 'score': 0.1785}, {'term': 